# Beyond Out-of-the-Box BQCA: Progressive 3-Layer Semantic Precision on BigQuery

### Overview
BigQuery Conversational Analytics (BQCA) provides powerful natural-language querying directly over enterprise data warehouses. However, naive out-of-the-box text-to-SQL poses significant risks for business-critical analytics: models hallucinate non-existent columns, generate silent fan-out joins, drop zero-count dimensions, and confidently state fabricated numbers.

This interactive walkthrough demonstrates a **Progressive 3-Layer Semantic Architecture** that elevates BQCA to enterprise-grade precision:
- **Layer 1: Schema & Graph Foundations** — Declarative BigQuery Property Graph `MEASURE()` declarations, non-enforced PK/FK constraints, and verified query starter templates.
- **Layer 2: Transport & Trust Evaluation** — 3-Tier stream trust classification catching assumption markers, ungrounded queries, and prose-to-row numeric reconciliation.
- **Layer 3: Deterministic Semantic Compiler & Policy Guardrails** — Code-decided Tier-1 compilation with zero-fanout `LEFT JOIN` precision, adversarial trap fallbacks, and dry-run safety gates.

---

### Prerequisites & Execution Modes
1. **Dependencies**:
   ```bash
   uv sync --extra advanced
   ```
2. **Interactive Jupyter Launch** (without modifying `uv.lock`):
   ```bash
   uv run --with jupyter jupyter lab notebooks/beyond_bqca_walkthrough.ipynb
   ```
3. **Execution Modes**:
   - **Part 1 (Sections 1-4)**: Runs **100% offline in <5 seconds** using bundled YAML contracts and wire response fixtures. Zero Google Cloud credentials or billing required.
   - **Part 2 (Section 5)**: Optionally executes live against `bigquery-public-data.thelook_ecommerce` when `RUN_LIVE_BQ=1` and `GOOGLE_CLOUD_PROJECT` are set in your environment.

## Section 1: Why Naive Text-to-SQL Fails Silently

When an analyst asks: *"What is our active buyer rate?"*, a naive text-to-SQL LLM typically generates:
```sql
SELECT
  COUNT(DISTINCT o.user_id) / COUNT(DISTINCT u.id) AS active_rate
FROM `bigquery-public-data.thelook_ecommerce.users` u
JOIN `bigquery-public-data.thelook_ecommerce.orders` o
  ON u.id = o.user_id;
```
This naive query fails silently in two critical ways:
1. **Inner Join Trivialization**: `JOIN orders` drops every user without an order from the denominator. Because `u.id = o.user_id` on every surviving joined row, `COUNT(DISTINCT o.user_id)` and `COUNT(DISTINCT u.id)` are identical, making the query trivially return `1.0` (`100.0%`) regardless of the actual user base.
2. **Missing Lifecycle Filter**: Even when switching to `LEFT JOIN`, an unconstrained join without `CASE WHEN o.status = 'Complete'` overcounts cancelled, returned, or pending buyers instead of completed buyers, violating the business definition of an active buyer.

A governed **Semantic Contract** (`config/semantic_contracts/*.yaml`) explicitly specifies the primary keys, join relationships, required filters, and numerator/denominator expressions.


In [ ]:
import os
import sys
from pathlib import Path

# Auto-resolve repository root regardless of current working directory
def find_repo_root() -> Path:
    env_root = os.getenv("BQCA_REPO_ROOT")
    if env_root and Path(env_root).is_dir():
        return Path(env_root).resolve()
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / "config" / "semantic_contracts").is_dir():
            return candidate.resolve()
        if (candidate / "bq_caapi_ge" / "config" / "semantic_contracts").is_dir():
            return (candidate / "bq_caapi_ge").resolve()
    try:
        import semantic
        return Path(semantic.__file__).resolve().parents[1]
    except ImportError:
        pass
    raise RuntimeError("Could not resolve bq_caapi_ge repository root.")

repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from semantic.registry import load_contracts  # noqa: E402

# Load governed semantic contracts
contracts = load_contracts(repo_root / "config" / "semantic_contracts")
contracts_by_id = {c.id: c for c in contracts}
print(f"Loaded {len(contracts)} semantic contracts: {list(contracts_by_id.keys())}")

orders_contract = contracts_by_id["thelook_orders"]
active_buyer_metric = orders_contract.metrics["active_buyer_rate"]

print(f"\nGoverned Metric: {active_buyer_metric.label} ({active_buyer_metric.name})")
print(f"  Type       : {active_buyer_metric.type}")
print(f"  Base Table : {active_buyer_metric.base_table}")
print(f"  Numerator  : {active_buyer_metric.numerator_sql}")
print(f"  Denominator: {active_buyer_metric.denominator_sql}")
print(f"  Join Path  : {active_buyer_metric.join_path}")

## Section 2: Layer 1 — Upgrade Native BQCA With GA BigQuery Graph `MEASURE()` & PK/FK Hints

BigQuery General Availability (GA) Property Graphs introduce `MEASURE()` properties on graph node tables. By exporting our semantic contracts into native BigQuery DDL, we provide native BQCA with:
1. **Property Graph DDL**: Declares node/edge relationships and single-table `MEASURE()` aggregations. Cross-table ratios (such as `active_buyer_rate`) and ranked metrics (such as `top_users_by_completed_revenue`) are safely detected and skipped from the graph DDL.
2. **Non-Enforced Constraints DDL**: Adds PK and FK constraints with `NOT ENFORCED` to inform the BigQuery query optimizer.
3. **Verified Queries**: Pre-compiled canonical SQL queries exported as starter templates.

In [ ]:
from semantic.graph_export import (
    export_property_graph_ddl,
    export_fk_constraints_ddl,
    export_verified_queries,
)

# 1. Export GA BigQuery Property Graph DDL with MEASURE() declarations
graph_res = export_property_graph_ddl(
    orders_contract,
    graph_project="my-analytical-project",
    graph_dataset="thelook_ecommerce",
    graph_name="thelook_orders_graph",
)

print("=== BigQuery Property Graph DDL Preview ===")
print(graph_res.ddl[:600] + "\n... [truncated for display]")

print(f"\nExported Single-Table Measures ({len(graph_res.exported_metrics)}):")
for m_id in graph_res.exported_metrics:
    print(f"  [+] {m_id}")

print(f"\nSkipped Complex/Cross-Table Metrics ({len(graph_res.skipped_metrics)}):")
for sm in graph_res.skipped_metrics:
    print(f"  [-] {sm.metric_id}: {sm.reason}")

# 2. Export Non-Enforced PK/FK Constraints
fk_ddl = export_fk_constraints_ddl(
    orders_contract,
    target_project="my-analytical-project",
    target_dataset="thelook_ecommerce",
)
print("\n=== Non-Enforced Foreign Key Constraints DDL Preview ===")
print(fk_ddl[:450] + "\n... [truncated for display]")

# 3. Export Starter Verified Query Specifications
verified_queries = export_verified_queries(orders_contract)
print(f"\n=== Compiled Verified Queries ({len(verified_queries)}) ===")
for vq in verified_queries:
    print(f"  - [{vq.query_id}] {vq.question}")

## Section 3: Layer 2 — 3-Tier Stream Trust & Numeric Reconciliation

Even with schema hints, native LLM streams can exhibit hallucinations or make ungrounded assumptions.
Layer 2 intercepts the raw token stream from Conversational Analytics and evaluates trust across 3 tiers:
- **Tier 1 (Verified / Compiled)**: Exact match against an authored verified query or certified compiler.
- **Tier 2 (Grounded)**: Generated valid SQL executed against allowed tables with numeric consistency.
- **Tier 3 (Review Required)**: Stream exhibits assumption markers (e.g. *"I assumed..."*), ungrounded prose, or out-of-scope refusals.

Additionally, `reconcile_prose_numbers` extracts all numerical claims in the model's generated text and asserts that every number exists in the query result rows.

In [ ]:
import json
from semantic.ca_trust import VerifiedQuerySpec, evaluate_ca_stream_trust
from semantic.numeric_reconcile import reconcile_prose_numbers

fixtures_dir = repo_root / "tests" / "fixtures" / "ca_streams"

# 1. Authored Verified Query Wire Fixture -> Tier 1 Verified
with open(fixtures_dir / "authored_verified_query.json", encoding="utf-8") as f:
    avq_data = json.load(f)

avq_spec = VerifiedQuerySpec(
    question="How many completed orders do we have?",
    sql="""SELECT COUNT(DISTINCT order_id) AS completed_order_count
FROM `example-project.example_dataset.orders`
WHERE status = 'completed'""",
    query_id="verified_completed_orders",
)
res_1 = evaluate_ca_stream_trust(
    avq_data["messages"],
    question=avq_data.get("question", "How many completed orders do we have?"),
    authored_verified_queries=(avq_spec,),
)
print(f"Fixture 1 (Authored Verified Query) -> Tier: {res_1.tier.value} ({res_1.sub_mode})")

# 2. Wire Fixture with Assumption Marker -> Tier 3 Review Required
with open(fixtures_dir / "trap_active_rate_assumption.json", encoding="utf-8") as f:
    trap_data = json.load(f)

res_2 = evaluate_ca_stream_trust(
    trap_data["messages"],
    question=trap_data.get("question", "What is our active buyer rate?"),
)
print(f"Fixture 2 (Trap Assumption Marker)  -> Tier: {res_2.tier.value} ({res_2.sub_mode})")
print(f"  Flagged Reasons: {res_2.reasons}")

# 3. Wire Fixture with Out-of-Scope Refusal -> Tier 3 Refusal
with open(fixtures_dir / "out_of_scope_refusal.json", encoding="utf-8") as f:
    refusal_data = json.load(f)

res_3 = evaluate_ca_stream_trust(
    refusal_data["messages"],
    question=refusal_data.get("question", "Can you help me with something else?"),
)
print(f"Fixture 3 (Out-of-Scope Refusal)    -> Tier: {res_3.tier.value} ({res_3.sub_mode})")

# 4. Numeric Reconciliation: Catching a Hallucinated Prose Number
hallucinated_prose = "Completed revenue was $199,999.00 across 42 orders."
actual_result_rows = [{"completed_revenue": 125430.50, "order_count": 42}]

unreconciled_numbers = reconcile_prose_numbers(
    hallucinated_prose,
    question="What was our completed revenue?",
    result_rows=actual_result_rows,
)
print("\n=== Prose-to-Row Numeric Reconciliation ===")
print(f"Model Prose : '{hallucinated_prose}'")
print(f"Result Rows : {actual_result_rows}")
print(f"Flagged Fabrications: {unreconciled_numbers}")
assert len(unreconciled_numbers) > 0

## Section 4: Layer 3 — Deterministic Tier-1 Compiler & Adversarial Trap Guardrails

For mission-critical business metrics, we eliminate LLM non-determinism entirely:
1. **Tier-1 Deterministic Compiler**: When the model selects intent from the semantic catalog, `try_compile_narrow_selection` emits certified GoogleSQL with fanout-free `LEFT JOIN` semantics.
2. **Adversarial Trap Fallback**: If the question introduces ambiguity, unmodeled metrics, or missing filter qualifiers, the compiler returns `None` to safely fall back to Tier-2 guarded generation rather than certifying incorrect SQL.
3. **Dry-Run Security Gates**: `validate_dry_run_summary` validates BigQuery dry-run metadata before execution, blocking non-SELECT statements (`DELETE`, `UPDATE`), unauthorized tables (`INFORMATION_SCHEMA`), and queries exceeding byte scan caps.

In [ ]:
from semantic.runtime import (
    SemanticConceptSelection,
    SemanticSelection,
    try_compile_narrow_selection,
)
from semantic.guardrails import (
    DryRunSummary,
    SqlPolicyError,
    validate_dry_run_summary,
)

# 1. Positive Compilation: active_buyer_rate by country
sel_active = SemanticSelection(
    reason="Query for active buyer rate by country.",
    selected_contexts=[
        SemanticConceptSelection(
            context_id="thelook_orders",
            context_version=1,
            metric_ids=["active_buyer_rate"],
            dimension_ids=["country"],
            group_by_dimension_ids=["country"],
        )
    ],
)
compiled_active = try_compile_narrow_selection(
    question="What is our active buyer rate by country?",
    selection=sel_active,
    contracts_by_id=contracts_by_id,
)
assert compiled_active is not None
print("=== Certified SQL: active_buyer_rate by country ===")
print(compiled_active["sql"])

# 2. Positive Compilation: completed_order_count
sel_orders = SemanticSelection(
    reason="Query for completed order count.",
    selected_contexts=[
        SemanticConceptSelection(
            context_id="thelook_orders",
            context_version=1,
            metric_ids=["completed_order_count"],
        )
    ],
)
compiled_orders = try_compile_narrow_selection(
    question="How many completed orders were placed?",
    selection=sel_orders,
    contracts_by_id=contracts_by_id,
)
assert compiled_orders is not None
print("\n=== Certified SQL: completed_order_count ===")
print(compiled_orders["sql"])

# 3. Deterministic Fallbacks on 4 Real Adversarial Traps
traps = [
    (
        "How many products were ordered?",
        SemanticSelection(
            reason="Trap 1: Question asks for products, but metric is completed orders.",
            selected_contexts=[
                SemanticConceptSelection(
                    context_id="thelook_orders",
                    context_version=1,
                    metric_ids=["completed_order_count"],
                )
            ],
        ),
    ),
    (
        "What is the total revenue?",
        SemanticSelection(
            reason="Trap 2: Question omits 'completed' required filter qualifier.",
            selected_contexts=[
                SemanticConceptSelection(
                    context_id="thelook_orders",
                    context_version=1,
                    metric_ids=["completed_revenue"],
                )
            ],
        ),
    ),
    (
        "How many completed orders per user?",
        SemanticSelection(
            reason="Trap 3: Grouping cue 'per user' does not match selected dimension 'country'.",
            selected_contexts=[
                SemanticConceptSelection(
                    context_id="thelook_orders",
                    context_version=1,
                    metric_ids=["completed_order_count"],
                    dimension_ids=["country"],
                    group_by_dimension_ids=["country"],
                )
            ],
        ),
    ),
    (
        "What is our customer churn rate?",
        SemanticSelection(
            reason="Trap 4: Customer churn rate is not declared in the contract.",
            selected_contexts=[
                SemanticConceptSelection(
                    context_id="thelook_orders",
                    context_version=1,
                    metric_ids=["active_buyer_rate"],
                )
            ],
        ),
    ),
]

print("\n=== Adversarial Trap Fallback Evaluation ===")
for q, sel in traps:
    res = try_compile_narrow_selection(question=q, selection=sel, contracts_by_id=contracts_by_id)
    status = "Fallback to Tier 2 (Safe)" if res is None else "Certified (Unexpected!)"
    print(f"  Trap: '{q}' -> {status}")
    assert res is None, f"Expected deterministic fallback for trap: {q}"

# 4. Dry-Run Policy Gate Violations
print("\n=== Dry-Run Policy Gate Verification ===")
# Block non-SELECT
try:
    validate_dry_run_summary(
        "DELETE FROM `my-project.thelook_ecommerce.users` WHERE TRUE",
        DryRunSummary(statement_type="DELETE"),
    )
except SqlPolicyError as e:
    print(f"  [+] Correctly blocked DML statement: {e}")

# Block unauthorized table (e.g. metadata snooping)
try:
    validate_dry_run_summary(
        "SELECT * FROM `region-us.INFORMATION_SCHEMA.JOBS_BY_PROJECT`",
        DryRunSummary(
            statement_type="SELECT",
            referenced_sources=frozenset(["region-us.INFORMATION_SCHEMA.JOBS_BY_PROJECT"]),
        ),
        allowed_sources=frozenset(["bigquery-public-data.thelook_ecommerce.users"]),
    )
except SqlPolicyError as e:
    print(f"  [+] Correctly blocked unauthorized source: {e}")

# Block scan exceeding byte budget
try:
    validate_dry_run_summary(
        "SELECT * FROM `bigquery-public-data.thelook_ecommerce.users`",
        DryRunSummary(
            statement_type="SELECT",
            referenced_sources=frozenset(["bigquery-public-data.thelook_ecommerce.users"]),
            total_bytes_processed=50_000_000,
        ),
        allowed_sources=frozenset(["bigquery-public-data.thelook_ecommerce.users"]),
        max_bytes_billed=10_000_000,
    )
except SqlPolicyError as e:
    print(f"  [+] Correctly blocked scan byte budget overrun: {e}")

## Section 5: Optional Live BigQuery & ADK Execution (`RUN_LIVE_BQ=1`)

This optional section connects to live BigQuery using Application Default Credentials (ADC) to run both queries against `bigquery-public-data.thelook_ecommerce` and measure the real-world discrepancy between naive text-to-SQL and governed compilation.

In [ ]:
if os.getenv("RUN_LIVE_BQ") == "1" and os.getenv("GOOGLE_CLOUD_PROJECT"):
    from google.cloud import bigquery
    from semantic.compiler import compile_metric_query

    client = bigquery.Client()

    # 1. Naive Inner-Join Query (trivially 100% due to dropping non-ordering users)
    naive_inner_sql = """
    SELECT
      ROUND(SAFE_DIVIDE(COUNT(DISTINCT o.user_id), NULLIF(COUNT(DISTINCT u.id), 0)) * 100, 2) AS rate_pct
    FROM `bigquery-public-data.thelook_ecommerce.users` AS u
    JOIN `bigquery-public-data.thelook_ecommerce.orders` AS o
      ON u.id = o.user_id;
    """
    naive_inner_rate = list(client.query(naive_inner_sql).result())[0]["rate_pct"]

    # 2. Naive Unfiltered Left-Join Query (overcounts cancelled/returned/pending orders)
    naive_left_sql = """
    SELECT
      ROUND(SAFE_DIVIDE(COUNT(DISTINCT o.user_id), NULLIF(COUNT(DISTINCT u.id), 0)) * 100, 2) AS rate_pct
    FROM `bigquery-public-data.thelook_ecommerce.users` AS u
    LEFT JOIN `bigquery-public-data.thelook_ecommerce.orders` AS o
      ON u.id = o.user_id;
    """
    naive_left_rate = list(client.query(naive_left_sql).result())[0]["rate_pct"]

    # 3. Governed Compiled Query (like-for-like overall scalar: LEFT JOIN + Complete status)
    compiled_scalar = compile_metric_query(
        orders_contract,
        metric_id="active_buyer_rate",
        group_by_dimension_ids=(),
    )
    gov_row = list(client.query(compiled_scalar.sql).result())[0]
    gov_rate = round(float(gov_row["active_buyer_rate"] or 0.0) * 100, 2)

    print(f"1. Naive INNER JOIN Rate       : {naive_inner_rate}% (Trivially 100.0% due to dropped non-buyers!)")
    print(f"2. Naive Unfiltered LEFT JOIN  : {naive_left_rate}% (Overcounts cancelled/returned orders)")
    print(f"3. Governed Compiled Metric    : {gov_rate}% (Precise like-for-like scalar with Complete status)")
else:
    print("Skipping live BigQuery execution (set RUN_LIVE_BQ=1 and GOOGLE_CLOUD_PROJECT=... to run live).")


## Summary: Choosing the Right Layer

| Layer | Technology | Best Used For | What It Solves |
| :--- | :--- | :--- | :--- |
| **Layer 1** | BigQuery Property Graph `MEASURE()` + PK/FK Hints | Exploratory & Self-Serve BI | Gives native BQCA relational graph topology and single-table aggregate measures. |
| **Layer 2** | Client-Side 3-Tier Stream Classifier & Reconciler | Evaluation, Testing & Auditing | Traps prompt assumptions, catches hallucinated numbers, and audits response fidelity. |
| **Layer 3** | Deterministic Semantic Compiler & Dry-Run Gates | Production & Regulated Analytics | Provides deterministic SQL compilation for covered contract metrics, zero join fan-out, and enforces single-statement SELECT-only policy, table allowlists, and scan byte caps. |
